# JPSS (Suomi NPP, NOAA-20, NOAA-21) / ATMS (Advanced Technology Microwave Sounder)

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=noaa/jpss-atms). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L1B

Level 1B (SDR): calibrated, geolocated antenna and brightness temperatures for 22 channels.

Looks for the ATMS product prefix in the public NOAA-21 bucket on AWS (no credentials), downloads the first file of a day and opens it. The layout is discovered at run time and the snippet explains what it found.

In [ ]:
%pip install -q s3fs xarray h5netcdf

In [ ]:
import s3fs
import xarray as xr

fs = s3fs.S3FileSystem(anon=True)                # public NOAA bucket, no credentials needed
BUCKET = "noaa-nesdis-n21-pds"                    # noaa-nesdis-n20-pds (NOAA-20), noaa-nesdis-snpp-pds (S-NPP, until 2026)
prefixes = fs.ls(BUCKET)
print(len(prefixes), "top-level product prefixes")
match = [p for p in prefixes if "atms" in p.lower()]
print(match)
if not match:
    raise SystemExit("No product prefix with 'atms': inspect the list above")

DAY = "2024/06/01"
files = fs.ls("%s/%s/" % (match[0], DAY))
print(len(files), "files on", DAY)
if not files:
    print(fs.ls(match[0])[:10])
    raise SystemExit("Nothing for this day: see the prefix listing above to adapt the date pattern")

fs.get(files[0], "data_" + files[0].split("/")[-1])
local = "data_" + files[0].split("/")[-1]
ds = xr.open_dataset(local, engine="h5netcdf")
print(ds)